# Unit 6 — Tool approval (human-in-the-loop) ⭐

`@tool(approval_mode="always_require")` makes the agent pause and ask before running the tool. This is the guardrail for anything dangerous.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from random import randint
from typing import Annotated, Any
from pydantic import Field
from agent_framework import AgentResponse, Message, tool
from agent_framework.openai import OpenAIChatCompletionClient

## Two tools: one safe, one gated

`get_weather` runs freely. `send_email` will pause.

In [ ]:
@tool(approval_mode="never_require")
def get_weather(city: Annotated[str, Field(description="City.")]) -> str:
    """Return the current weather."""
    return f"{city}: sunny, {randint(15, 25)}°C"

@tool(approval_mode="always_require")
def send_email(to: Annotated[str, Field(description="Recipient.")],
               subject: Annotated[str, Field(description="Subject.")],
               body: Annotated[str, Field(description="Body.")]) -> str:
    """Send an email. Requires user approval."""
    print(f"[fake-mailer] to={to} subject={subject}\n  body: {body}")
    return f"email sent to {to}"

In [ ]:
agent = OpenAIChatCompletionClient().as_agent(
    name="Picnicker",
    instructions="You help plan picnics and send update emails. Check the weather first.",
    tools=[get_weather, send_email],
)
session = agent.create_session()

## The approval loop

After `agent.run(...)`, check `result.user_input_requests`. If non-empty, the agent paused. Ask your human, feed the approval responses back, and `agent.run(...)` again — **with the same session** so the agent remembers the earlier tool call it was about to make. Without the session it would replan from scratch each round and keep re-asking.

In [ ]:
import asyncio

async def handle_approvals(query: str, agent, session, max_rounds: int = 10) -> AgentResponse:
    result = await agent.run(query, session=session)
    rounds = 0
    while result.user_input_requests:
        rounds += 1
        if rounds > max_rounds:
            raise RuntimeError(f"Exceeded {max_rounds} approval rounds")
        responses: list[Any] = []
        for request in result.user_input_requests:
            fn = request.function_call
            print(f"\nApproval requested:")
            print(f"  function: {fn.name}")
            print(f"  args:     {fn.arguments}")
            answer = await asyncio.to_thread(input, "Approve? (y/n): ")
            approved = answer.strip().lower().startswith("y")
            responses.append(Message("user", [request.to_function_approval_response(approved)]))
        result = await agent.run(responses, session=session)
    return result

result = await handle_approvals(
    "Email alex@example.com that the weather in Vienna looks great for the picnic. Check the weather first.",
    agent, session,
)
print("\nAgent:", result.text)

## Modes to know

- `"never_require"` — runs freely (default).
- `"always_require"` — pause on every call.

The 1.1 API only has these two modes. To ask-once-per-session, track approved function+args in your own `handle_approvals` loop.